# 13. Machine Learning with scikit-learn -- Predicting Polymer Properties

In Notebook 12 (Fitting) you learned to fit a function with a **known** form (e.g. Gaussian, exponential) to data. Machine learning is fitting when you do **not** know the functional form -- the algorithm learns the shape of the relationship from the data.

In this notebook we apply ML to the Kaggle polymer challenge: **predicting FFV (Fractional Free Volume) from molecular features**.

**Topics covered:**
- The supervised learning workflow: features, targets, train/test split
- Linear Regression
- Random Forest
- Evaluation metrics (MAE, RMSE, R-squared)
- Overfitting and generalisation
- Feature engineering impact

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## 13.1 What Is Machine Learning?

**Supervised learning** in one sentence:

> Given a table of features $X$ and a target variable $y$, learn a function $f$ such that $y \approx f(X)$.

Two types:
- **Regression**: the target is a continuous number (e.g. FFV = 0.374)
- **Classification**: the target is a category (e.g. "toxic" / "non-toxic")

We are doing **regression** today.

---
## 13.2 Preparing the Data

We use the artifact from CodingTask 1 that already has a train/validation split:

In [ ]:
df = pd.read_csv("artifacts/ffv_ready_split.csv")
print(f"Total samples: {len(df)}")
print(f"Split distribution:\n{df['split'].value_counts()}")
df.head()

In [ ]:
# Define features and target
feature_cols = ["smiles_len", "count_star", "count_C", "count_O", "count_N", "count_equal"]
target_col = "FFV"

# Split into train and validation sets
train_mask = df["split"] == "train"
valid_mask = df["split"] == "valid"

X_train = df.loc[train_mask, feature_cols].values
y_train = df.loc[train_mask, target_col].values
X_valid = df.loc[valid_mask, feature_cols].values
y_valid = df.loc[valid_mask, target_col].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_valid shape: {X_valid.shape}")
print(f"y_valid shape: {y_valid.shape}")

Notice: `.values` gives us NumPy arrays -- exactly what scikit-learn expects.

### Feature standardisation

Many ML algorithms work better when features are on the same scale. `StandardScaler` subtracts the mean and divides by the standard deviation -- exactly what we did manually in Notebook 11!

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_train)  # compute mean and std from training data ONLY

X_train_s = scaler.transform(X_train)
X_valid_s = scaler.transform(X_valid)

print("Before scaling - first row:", X_train[0])
print("After scaling  - first row:", X_train_s[0].round(3))

> **Why fit on train only?** If we compute the mean and std using the validation data too, we "leak" information from the validation set into the preprocessing step. This is called **data leakage** and it makes our evaluation overly optimistic.

---
## 13.3 First Model: Linear Regression

A linear regression predicts:

$$\hat{y} = w_1 x_1 + w_2 x_2 + \ldots + w_6 x_6 + b$$

This is exactly `X @ w + b` from Notebook 11 -- but now scikit-learn finds the **optimal** weights.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_s, y_train)  # That's it -- one line to train!

print("Coefficients (weights):", lr.coef_.round(4))
print("Intercept (bias):      ", round(lr.intercept_, 4))

In [ ]:
# Predict on the validation set
y_pred_lr = lr.predict(X_valid_s)

### Evaluation metrics

How good are our predictions? We use three common regression metrics:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_lr = mean_absolute_error(y_valid, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_valid, y_pred_lr))
r2_lr = r2_score(y_valid, y_pred_lr)

print(f"Linear Regression Results:")
print(f"  MAE:       {mae_lr:.4f}  (average absolute error)")
print(f"  RMSE:      {rmse_lr:.4f}  (penalises large errors more)")
print(f"  R-squared: {r2_lr:.4f}  (1.0 = perfect, 0.0 = baseline)")

### Visualising predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Predicted vs Actual
axes[0].scatter(y_valid, y_pred_lr, alpha=0.3, s=5)
lims = [y_valid.min(), y_valid.max()]
axes[0].plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
axes[0].set_xlabel("Actual FFV")
axes[0].set_ylabel("Predicted FFV")
axes[0].set_title(f"Linear Regression (R²={r2_lr:.3f})")
axes[0].legend()

# Residuals
residuals = y_valid - y_pred_lr
axes[1].scatter(y_pred_lr, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color="r", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicted FFV")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()

### Which features matter most?

In [ ]:
# Bar plot of coefficients
plt.figure(figsize=(8, 4))
plt.barh(feature_cols, lr.coef_)
plt.xlabel("Coefficient value")
plt.title("Linear Regression coefficients (standardised features)")
plt.tight_layout()
plt.show()

### Exercise 1

1. Which feature has the **largest absolute coefficient**? Does that make chemical sense?
2. What does a positive vs. negative coefficient mean in terms of FFV?

In [ ]:
# Your code / answer here


---
## 13.4 Second Model: Random Forest

A **Random Forest** builds many decision trees, each trained on a random subset of the data and features, and then averages their predictions. This allows it to capture **non-linear** relationships that linear regression cannot.

Intuition: imagine asking 100 chemists to estimate the FFV, each with a slightly different view of the data. The average of their answers is often better than any single answer.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random forests don't need feature standardisation, but it doesn't hurt
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_valid)

In [ ]:
mae_rf = mean_absolute_error(y_valid, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_valid, y_pred_rf))
r2_rf = r2_score(y_valid, y_pred_rf)

print(f"Random Forest Results:")
print(f"  MAE:       {mae_rf:.4f}")
print(f"  RMSE:      {rmse_rf:.4f}")
print(f"  R-squared: {r2_rf:.4f}")

### Comparing both models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R²": [r2_lr, r2_rf]
})
results.round(4)

In [ ]:
# Side-by-side scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, name, r2 in [
    (axes[0], y_pred_lr, "Linear Regression", r2_lr),
    (axes[1], y_pred_rf, "Random Forest", r2_rf)
]:
    ax.scatter(y_valid, y_pred, alpha=0.3, s=5)
    lims = [y_valid.min(), y_valid.max()]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel("Actual FFV")
    ax.set_ylabel("Predicted FFV")
    ax.set_title(f"{name} (R²={r2:.3f})")

plt.tight_layout()
plt.show()

### Feature importance

In [ ]:
importances = rf.feature_importances_

plt.figure(figsize=(8, 4))
plt.barh(feature_cols, importances)
plt.xlabel("Feature importance")
plt.title("Random Forest feature importances")
plt.tight_layout()
plt.show()

### Exercise 2

1. Try training a Random Forest with `n_estimators=10` and `n_estimators=200`. How do the results change?
2. Does feature importance agree with the Linear Regression coefficients?

In [ ]:
# Your code here


---
## 13.5 Overfitting: The Most Important Concept in ML

**Overfitting** happens when a model memorises the training data instead of learning generalisable patterns. The symptom: great performance on training data, poor performance on validation data.

Let's demonstrate with the Random Forest's `max_depth` parameter:

In [ ]:
depths = [2, 5, 10, 15, 20, 30, None]  # None = no limit
train_scores = []
valid_scores = []

for d in depths:
    rf_temp = RandomForestRegressor(n_estimators=50, max_depth=d, random_state=42)
    rf_temp.fit(X_train, y_train)
    train_scores.append(r2_score(y_train, rf_temp.predict(X_train)))
    valid_scores.append(r2_score(y_valid, rf_temp.predict(X_valid)))

# Replace None with a number for plotting
depth_labels = [str(d) if d is not None else "None" for d in depths]

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depth_labels, train_scores, "o-", label="Train R²", color="steelblue")
plt.plot(depth_labels, valid_scores, "s-", label="Validation R²", color="coral")
plt.xlabel("max_depth")
plt.ylabel("R² score")
plt.title("Overfitting: the gap between train and validation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

When `max_depth` is too high (or unlimited), the training R² approaches 1.0 but the validation R² may plateau or even decrease. The model is **overfitting**.

The goal is to find a balance where the validation score is maximised -- this is called **model selection**.

### Exercise 3

1. From the plot above, which `max_depth` gives the best validation R²?
2. At which depth does overfitting become clearly visible (large train-validation gap)?

In [ ]:
# Your answer here


---
## 13.6 Adding RDKit Features

In Notebook 12 we saw that RDKit can compute molecular descriptors like molecular weight and TPSA. In CodingTask 1, some of you added these as bonus features. Let's see if they improve the model:

In [ ]:
df_rdkit = pd.read_csv("artifacts/ffv_features_rdkit.csv")
df_rdkit.head()

In [ ]:
# Extended feature set: basic + RDKit descriptors
extended_cols = feature_cols + ["MolWt", "NumValenceElectrons", "TPSA"]

X_train_ext = df_rdkit.loc[df_rdkit["split"] == "train", extended_cols].values
X_valid_ext = df_rdkit.loc[df_rdkit["split"] == "valid", extended_cols].values

print(f"Extended features: {X_train_ext.shape[1]} columns (was {X_train.shape[1]})")

In [ ]:
# Train a Random Forest with the extended features
rf_ext = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf_ext.fit(X_train_ext, y_train)
y_pred_ext = rf_ext.predict(X_valid_ext)

r2_ext = r2_score(y_valid, y_pred_ext)
mae_ext = mean_absolute_error(y_valid, y_pred_ext)

print(f"\nResults comparison:")
print(f"  Random Forest (6 features): R²={r2_rf:.4f}, MAE={mae_rf:.4f}")
print(f"  Random Forest (9 features): R²={r2_ext:.4f}, MAE={mae_ext:.4f}")

In [ ]:
# Feature importance with extended features
plt.figure(figsize=(8, 5))
plt.barh(extended_cols, rf_ext.feature_importances_)
plt.xlabel("Feature importance")
plt.title("Random Forest feature importances (extended features)")
plt.tight_layout()
plt.show()

> **Key takeaway:** Better features matter more than fancier models. This is called **feature engineering**, and it is one of the most important skills in data science.

---
## 13.7 Summary

| Step | What we did |
|------|-------------|
| Prepare data | Load CSV, extract X/y arrays, train/validation split |
| Standardise | `StandardScaler` -- fit on train, transform both |
| Linear Regression | `LinearRegression().fit(X, y)` -- one line |
| Random Forest | `RandomForestRegressor().fit(X, y)` -- handles non-linearity |
| Evaluate | MAE, RMSE, R² -- always on the **validation** set |
| Overfitting | Train vs validation gap -- control with `max_depth` |
| Feature engineering | Adding RDKit descriptors improved the model |

**Next up:** In Notebook 14, we get a glimpse of deep learning -- can a neural network do even better?